In [11]:
from pathlib import Path ## Rutas relativas
from PIL import Image  ## Abrir y manipular imagenes
import numpy as np ### Procesar Matrices - Valores de pixeles
import matplotlib.pyplot as plt ## Graficar

In [23]:
IMAGE_DIR = Path("data/images")  ##Ruta a la carpeta donde guardamos las imagenes

image_paths = sorted(
    path for path in IMAGE_DIR.glob("*") ## Lista de rutas de archivos
    if path.is_file()
)

print(f"Cant. de imagenes: {len(image_paths)}")

Cant. de imagenes: 10


#### Conocer las imagenes

In [28]:
for path in image_paths:
    if path.is_file():
        image = Image.open(path)

        width, height = image.size

        print(f"{path.name}: tamaño={width} × {height}")

albert-bloch_figures-in-silver-light.jpg: tamaño=1657 × 1382
albert-gleizes_femme-cubiste-1921.jpg: tamaño=1382 × 1889
aldemir-martins_baiana-1980.jpg: tamaño=1382 × 1845
aldo-mondino_collage-1973.jpg: tamaño=1382 × 1742
aleksey-savrasov_early-spring-2.jpg: tamaño=1880 × 1382
alvaro-lapa_c-line-s-notebook-1990.jpg: tamaño=3300 × 1382
andre-derain_the-port-of-collioure-1905.jpg: tamaño=1695 × 1382
betty-parsons_ladder-1968.jpg: tamaño=1382 × 1949
joshua-reynolds_jane-fleming-later-countess-of-harrington-1779.jpg: tamaño=1382 × 2289
yayoi-kusama_fields-in-spring-1988.jpg: tamaño=1382 × 1629


### Redimensionamiento de las imágenes

Las imágenes seleccionadas tienen dimensiones variables, por ejemplo, `3300 × 1382` y `1382 × 1629`. Por esta razón, se realizará un proceso de redimensionamiento (*resize*) antes de utilizarlas en el modelo.

No se utilizará un tamaño fijo para todas las imágenes, ya que esto podría cambiar su relación de aspecto y generar deformaciones. En su lugar, se establecerá un tamaño máximo y cada imagen se redimensionará manteniendo su proporción original.

El objetivo del redimensionamiento es **reducir el número de píxeles que debe procesar K-Means**, haciendo que el algoritmo sea más rápido y requiera menos recursos, pero conservando la información de color de las obras.

Aunque en este ejercicio se trabaja con un número reducido de imágenes, algunas de ellas tienen millones de píxeles. Además, el pipeline debe estar preparado para recibir imágenes nuevas que podrían tener una resolución mucho mayor. Por esta razón, se incluye el redimensionamiento como parte del preprocesamiento, buscando reducir el costo computacional sin perder información relevante para la extracción de la paleta de colores.



### Decisión sobre el factor de reducción

En lugar de establecer un tamaño fijo para todas las imágenes, se decidió evaluar diferentes **factores de reducción**. Esta decisión permite mantener la proporción original de cada imagen y, al mismo tiempo, analizar cómo la reducción del número de píxeles afecta la información de color y el costo computacional del procesamiento con K-Means. Se probarán los factores **1, 1/2, 1/4, 1/8, 1/16 y 1/32**, donde cada factor se aplica tanto al ancho como al alto de la imagen. De esta forma, la selección del factor final no se realizará de manera arbitraria, sino a partir de la comparación de los resultados obtenidos en términos de conservación de la información y eficiencia computacional.


In [32]:
RESIZE_FACTORS = [1, 1/2, 1/3, 1/4, 1/8, 1/16, 1/32]

def calculate_resize_size(width, height, factor):
    new_width = int(width * factor)
    new_height = int(height * factor)

    return new_width, new_height



In [33]:
for path in image_paths:
    if path.is_file():
        image = Image.open(path)

        width, height = image.size

        print(f"\n{path.name}")
        print(f"Original: {width} × {height}")

        for factor in RESIZE_FACTORS:
            new_width, new_height = calculate_resize_size(
                width, height, factor
            )

            print(
                f"Factor {factor}: "
                f"{new_width} × {new_height}"
            )


albert-bloch_figures-in-silver-light.jpg
Original: 1657 × 1382
Factor 1: 1657 × 1382
Factor 0.5: 828 × 691
Factor 0.3333333333333333: 552 × 460
Factor 0.25: 414 × 345
Factor 0.125: 207 × 172
Factor 0.0625: 103 × 86
Factor 0.03125: 51 × 43

albert-gleizes_femme-cubiste-1921.jpg
Original: 1382 × 1889
Factor 1: 1382 × 1889
Factor 0.5: 691 × 944
Factor 0.3333333333333333: 460 × 629
Factor 0.25: 345 × 472
Factor 0.125: 172 × 236
Factor 0.0625: 86 × 118
Factor 0.03125: 43 × 59

aldemir-martins_baiana-1980.jpg
Original: 1382 × 1845
Factor 1: 1382 × 1845
Factor 0.5: 691 × 922
Factor 0.3333333333333333: 460 × 615
Factor 0.25: 345 × 461
Factor 0.125: 172 × 230
Factor 0.0625: 86 × 115
Factor 0.03125: 43 × 57

aldo-mondino_collage-1973.jpg
Original: 1382 × 1742
Factor 1: 1382 × 1742
Factor 0.5: 691 × 871
Factor 0.3333333333333333: 460 × 580
Factor 0.25: 345 × 435
Factor 0.125: 172 × 217
Factor 0.0625: 86 × 108
Factor 0.03125: 43 × 54

aleksey-savrasov_early-spring-2.jpg
Original: 1880 × 1382
Fact

In [34]:
%pip install opencv-python

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
    --------------------------------------- 0.8/44.0 MB 2.9 MB/s eta 0:00:16
   --- ------------------------------------ 3.4/44.0 MB 8.1 MB/s eta 0:00:05
   ------- -------------------------------- 8.7/44.0 MB 14.1 MB/s eta 0:00:03
   ------------- -------------------------- 15.2/44.0 MB 18.7 MB/s eta 0:00:02
   -------------------- ------------------- 22.3/44.0 MB 22.0 MB/s eta 0:00:01
   ------------------------- -------------- 28.0/44.0 MB 22.8 MB/s eta 0:00:01
   ------------------------------ --------- 33.6/44.0 MB 23.6 MB/s eta 0:00:01
   ---------------------------------- ----- 37.5/44.0 MB 23.3 MB/s eta 0:00:01
   ------------------------------------ --- 40.1/44.0 MB 21.9 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 22.3 MB/s  0:00:02
Note: you may need to restart the kernel to use updated packages.


Todas las imágenes utilizadas actualmente se encuentran en formato RGB. Por lo tanto, no sería necesario realizar una conversión de color sobre este conjunto de datos. Sin embargo, se incluirá la conversión a RGB como parte del pipeline de preprocesamiento, con el propósito de garantizar que cualquier imagen incorporada posteriormente sea transformada a una representación consistente de tres canales antes de ingresar al modelo.

In [25]:
images_rgb = []

for path in image_paths:
    if path.is_file():
        image = Image.open(path)
        image_rgb = image.convert("RGB")
        images_rgb.append(image_rgb)

for path, image in zip(image_paths, images_rgb):
    print(path.name, image.mode)

albert-bloch_figures-in-silver-light.jpg RGB
albert-gleizes_femme-cubiste-1921.jpg RGB
aldemir-martins_baiana-1980.jpg RGB
aldo-mondino_collage-1973.jpg RGB
aleksey-savrasov_early-spring-2.jpg RGB
alvaro-lapa_c-line-s-notebook-1990.jpg RGB
andre-derain_the-port-of-collioure-1905.jpg RGB
betty-parsons_ladder-1968.jpg RGB
joshua-reynolds_jane-fleming-later-countess-of-harrington-1779.jpg RGB
yayoi-kusama_fields-in-spring-1988.jpg RGB
